# LLM Evaluation for Greenwashing Detection

This notebook combines extracted claims and KPIs to prompt an LLM and identify potential inconsistencies.

## Setup

In [ ]:
%pip install -q transformers accelerate pandas

## Imports

In [ ]:
from pathlib import Path
from typing import Dict

import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

## Configuration

In [ ]:
CLAIMS_CSV = Path('claims_extracted.csv')
KPIS_CSV = Path('kpis_extracted.csv')
MODEL_NAME = 'deepseek-ai/DeepSeek-R1-Zero'
MAX_NEW_TOKENS = 512

## Load Data

In [ ]:
claims_df = pd.read_csv(CLAIMS_CSV)
kpis_df = pd.read_csv(KPIS_CSV)

claims_df.head(), kpis_df.head()

## Prompt Construction

In [ ]:
def build_prompt(claim_row: pd.Series, kpi_row: pd.Series) -> str:
    claim_text = claim_row['text']
    claim_score = claim_row.get('score', float('nan'))
    kpi_values = kpi_row.drop(labels=['report']).to_dict()
    kpi_lines = '\n'.join(f"- {name}: {value}" for name, value in kpi_values.items() if value)
    prompt = (
        "You are an ESG auditor. Analyse the sustainability claim against the financial indicators.\n\n"
        f"Claim (confidence {claim_score:.2f}): {claim_text}\n\n"
        f"Financial indicators from annual report:\n{kpi_lines if kpi_lines else '- No KPIs found.'}\n\n"
        "Assess whether the claim is supported by the KPIs, flag inconsistencies, and explain your reasoning."
    )
    return prompt

## Load Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map='auto', torch_dtype='auto')
text_generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device_map='auto')

## Run Evaluation

In [ ]:
evaluations = []
for _, claim_row in claims_df.iterrows():
    base_name = claim_row['report'].split('.')[0]
    matching = kpis_df[kpis_df['report'].str.contains(base_name, case=False, na=False)]
    if matching.empty:
        matching = kpis_df
    for _, kpi_row in matching.iterrows():
        prompt = build_prompt(claim_row, kpi_row)
        response = text_generator(prompt, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)[0]['generated_text']
        evaluations.append({
            'claim_report': claim_row['report'],
            'kpi_report': kpi_row['report'],
            'claim_text': claim_row['text'],
            'kpis': kpi_row.to_dict(),
            'llm_response': response,
        })

evaluations_df = pd.DataFrame(evaluations)
evaluations_df.head()

## Save Evaluations

In [ ]:
evaluations_df.to_csv('llm_evaluations.csv', index=False)

The generated evaluations are stored in `llm_evaluations.csv` for further analysis.